# 02 - Decomposition Starter: RAG từng bước

Notebook này chia pipeline RAG cơ bản thành các bước nhỏ:
1. Import & cấu hình
2. Khởi tạo LLM
3. Load tài liệu
4. Split tài liệu
5. Vector store & retriever
6. Prompt
7. Hàm `query()` và test

In [2]:
import warnings

from dotenv import load_dotenv

from langchain_community.document_loaders import WebBaseLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.output_parsers import StrOutputParser

from langchain_core.prompts import PromptTemplate

from langchain_community.vectorstores import Chroma

from langchain_openai import ChatOpenAI, OpenAIEmbeddings



warnings.filterwarnings("ignore")

load_dotenv()  # đọc OPENAI_API_KEY từ .env

True

## Bước 1: Khởi tạo LLM

In [15]:
llm = ChatOpenAI(model="gpt-4o-mini")

## Bước 2: Load tài liệu

In [16]:
urls = [

    "https://vnexpress.net/thong-diep-5-tu-cua-elon-musk-khi-jensen-huang-phuc-chuc-5148278.html",

    "https://vnexpress.net/intel-sa-thai-mot-phan-nhan-su-it-toan-cau-5147238.html",

]

docs = []

for url in urls:

    docs.extend(WebBaseLoader(url).load())



for i, doc in enumerate(docs):
    with open(f"document_{i+1}.md", "w", encoding="utf-8") as f:
        f.write(doc.page_content)

## Bước 3: Split tài liệu

In [18]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(

    chunk_size=500, chunk_overlap=50

)

splits = text_splitter.split_documents(docs)

print(f"Số chunk: {len(splits)}")

Số chunk: 21


## Bước 4: Vector store & retriever

In [19]:
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=OpenAIEmbeddings(),
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever()



results = retriever.invoke("Elon Musk nói gì?")

for r in results:

    print(r.page_content[:150].replace("\n", " "), "...")

Elon Musk phản hồi ngắn gọn từ sau khi nhà đồng sáng lập Nvidia Jensen Huang khôi phục chức vụ CEO. "Sai lầm đã được sửa", Musk viết trên X ngày 23/11 ...
Elon Musk phản hồi ngắn gọn từ sau khi nhà đồng sáng lập Nvidia Jensen Huang khôi phục chức vụ CEO. "Sai lầm đã được sửa", Musk viết trên X ngày 23/11 ...
Thông điệp 5 từ của Elon Musk khi Jensen Huang phục chức - Báo VnExpress                                                                               ...
yêu quý ông".Musk cũng tham gia cùng gần 200.000 người ký tên trên một website kêu gọi "đưa Jensen trở lại". Ngày 11/11, vài tiếng trước khi Huang thô ...


## Bước 5: Prompt và hàm `query()`

In [22]:
prompt = PromptTemplate.from_template(

    "Answer the question using only the context below.\n\n"

    "Context:\n{context}\n\nQuestion: {question}\nAnswer:"

)





def query(question: str) -> str:

    context_docs = retriever.invoke(question)

    context = "\n\n".join(d.page_content for d in context_docs)

    chain = prompt | llm | StrOutputParser()

    return chain.invoke({"context": context, "question": question})

## Bước 6: Test

In [23]:
print(query("Intel đã làm gì?"))

Intel sa thải một phần nhân sự IT toàn cầu.
